# Partie I — MLP sur Spotify Tracks Dataset

Notebook séparé pour la **Partie I** du projet de Deep Learning.

Objectif : réaliser une classification supervisée sur des données tabulaires musicales avec un MLP sous PyTorch.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Projet_Deep_Learning_Musique"
os.makedirs(PROJECT_DIR, exist_ok=True)

DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

for folder in [DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Dossier projet :", PROJECT_DIR)

## 1. Imports et configuration

Cette section importe les bibliothèques nécessaires, fixe une seed pour la reproductibilité et sélectionne automatiquement le GPU si disponible.

In [ ]:
import os
import glob
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

required_vars = ["PROJECT_DIR", "DATA_DIR", "MODEL_DIR", "RESULTS_DIR"]
missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    raise NameError(
        "Les variables suivantes doivent déjà exister dans le notebook : "
        + ", ".join(missing_vars)
    )

SPOTIFY_DIR = os.path.join(DATA_DIR, "spotify")
print("Dossier Spotify :", SPOTIFY_DIR)

## 2. Vérification du dataset Spotify

In [ ]:
if not os.path.exists(SPOTIFY_DIR):
    raise FileNotFoundError(f"Le dossier Spotify est introuvable : {SPOTIFY_DIR}")

print("Contenu du dossier Spotify :")
for item in os.listdir(SPOTIFY_DIR):
    print("-", item)

csv_files = glob.glob(os.path.join(SPOTIFY_DIR, "*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError("Aucun fichier CSV trouvé dans le dossier Spotify.")

if len(csv_files) > 1:
    print("\nPlusieurs fichiers CSV trouvés :")
    for file in csv_files:
        print("-", file)

def csv_score(path):
    name = os.path.basename(path).lower()
    score = 0

    if "spotify" in name:
        score += 3
    if "track" in name or "tracks" in name:
        score += 3
    if "dataset" in name:
        score += 1

    size = os.path.getsize(path)
    return score, size

csv_path = sorted(csv_files, key=csv_score, reverse=True)[0]

print("\nFichier CSV sélectionné automatiquement :")
print(csv_path)

df = pd.read_csv(csv_path)

print("\nAperçu des 5 premières lignes :")
display(df.head())

print("\nTaille du dataframe :", df.shape)

print("\nColonnes disponibles :")
print(df.columns.tolist())

print("\nTypes de données :")
display(df.dtypes)

print("\nValeurs manquantes par colonne :")
display(df.isnull().sum())

if "track_genre" in df.columns:
    print("\nDistribution de la colonne cible track_genre :")
    display(df["track_genre"].value_counts())
else:
    print("\nAttention : la colonne cible track_genre n'existe pas dans ce fichier.")

## 3. Préparation des données

In [ ]:
target_column = "track_genre"

if target_column not in df.columns:
    raise ValueError(
        "La colonne track_genre est absente. "
        "Impossible de faire une classification supervisée par genre."
    )

top_n_genres = 10
genre_counts = df[target_column].value_counts()

if len(genre_counts) > top_n_genres:
    top_genres = genre_counts.head(top_n_genres).index
    df_work = df[df[target_column].isin(top_genres)].copy()
    print(f"Nombre de genres initial : {len(genre_counts)}")
    print(f"On garde les {top_n_genres} genres les plus fréquents.")
else:
    df_work = df.copy()
    print(f"Nombre de genres : {len(genre_counts)}")

candidate_features = [
    "popularity",
    "duration_ms",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

feature_columns = [col for col in candidate_features if col in df_work.columns]

if len(feature_columns) == 0:
    raise ValueError("Aucune colonne numérique utile n'a été trouvée.")

print("Colonnes utilisées comme features :")
print(feature_columns)

df_model = df_work[feature_columns + [target_column]].copy()
df_model = df_model.dropna(subset=feature_columns + [target_column])

print("Taille après suppression des valeurs manquantes :", df_model.shape)

X = df_model[feature_columns].values
y_text = df_model[target_column].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

num_classes = len(label_encoder.classes_)
input_dim = X.shape[1]

print("Nombre de features :", input_dim)
print("Nombre de classes :", num_classes)
print("Classes :", label_encoder.classes_)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

try:
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_scaled,
        y,
        test_size=0.30,
        random_state=SEED,
        stratify=y
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=SEED,
        stratify=y_temp
    )

except ValueError as error:
    print("Stratification impossible, division sans stratify.")
    print("Message :", error)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X_scaled,
        y,
        test_size=0.30,
        random_state=SEED
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=SEED
    )

print("Train :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("DataLoaders créés avec batch_size =", batch_size)

## 4. Modèles MLP

In [ ]:
class MLPSequential(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)


class MLPCustom(nn.Module):
    # nn.Module est la classe de base pour créer un modèle PyTorch.
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        return x

## 5. Initialisation des poids

In [ ]:
def init_weights(model, method="xavier"):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            if method == "gaussian":
                nn.init.normal_(module.weight, mean=0.0, std=0.01)

            elif method == "constant":
                nn.init.constant_(module.weight, 0.01)

            elif method == "xavier":
                nn.init.xavier_uniform_(module.weight)

            else:
                raise ValueError(
                    "Méthode inconnue. Choisir parmi : gaussian, constant, xavier."
                )

            nn.init.constant_(module.bias, 0.0)

    return model

## 6. Fonctions d'entraînement et d'évaluation

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_predictions = []
    all_labels = []

    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)

        predictions = torch.argmax(outputs, dim=1)
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_accuracy = accuracy_score(all_labels, all_predictions)

    return epoch_loss, epoch_accuracy


def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for features, labels in dataloader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * features.size(0)

            predictions = torch.argmax(outputs, dim=1)
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_accuracy = accuracy_score(all_labels, all_predictions)

    return epoch_loss, epoch_accuracy, np.array(all_predictions), np.array(all_labels)


def plot_training_curves(history, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train loss")
    plt.plot(epochs, history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Courbe de loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_accuracy"], label="Train accuracy")
    plt.plot(epochs, history["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Courbe d'accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()

    print("Courbe sauvegardée :", save_path)

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=20,
    patience=5,
    save_path=None
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    best_val_loss = np.inf
    best_val_accuracy = 0.0
    epochs_without_improvement = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_accuracy, _, _ = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val accuracy: {val_accuracy:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_accuracy = val_accuracy
            epochs_without_improvement = 0

            if save_path is not None:
                # On sauvegarde le meilleur modèle pour éviter de garder un modèle moins performant.
                torch.save(model.state_dict(), save_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping après {epoch} epochs.")
            break

    return history, best_val_loss, best_val_accuracy

## 7. Expériences comparatives

In [ ]:
experiments = [
    ("MLPSequential", MLPSequential, "gaussian"),
    ("MLPSequential", MLPSequential, "constant"),
    ("MLPSequential", MLPSequential, "xavier"),
    ("MLPCustom", MLPCustom, "gaussian"),
    ("MLPCustom", MLPCustom, "constant"),
    ("MLPCustom", MLPCustom, "xavier"),
]

criterion = nn.CrossEntropyLoss()
learning_rate = 0.001
num_epochs = 20
patience = 5

experiment_results = []

for model_name, model_class, init_method in experiments:
    experiment_name = f"{model_name}_{init_method}"

    print("\n" + "=" * 60)
    print("Expérience :", experiment_name)
    print("=" * 60)

    model = model_class(input_dim=input_dim, num_classes=num_classes)
    model = init_weights(model, method=init_method)
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model_save_path = os.path.join(MODEL_DIR, f"{experiment_name}_best.pt")
    curve_save_path = os.path.join(RESULTS_DIR, f"{experiment_name}_curves.png")

    history, best_val_loss, best_val_accuracy = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
        patience=patience,
        save_path=model_save_path
    )

    plot_training_curves(history, curve_save_path)

    experiment_results.append({
        "experiment": experiment_name,
        "model_name": model_name,
        "initialization": init_method,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": best_val_accuracy,
        "model_path": model_save_path,
        "curve_path": curve_save_path,
        "epochs_trained": len(history["train_loss"])
    })

results_df = pd.DataFrame(experiment_results)
display(results_df)

## 8. Évaluation finale

In [ ]:
results_csv_path = os.path.join(RESULTS_DIR, "mlp_spotify_experiments_results.csv")
results_df.to_csv(results_csv_path, index=False)

print("Tableau comparatif sauvegardé :", results_csv_path)

best_index = results_df.sort_values(
    by=["best_val_accuracy", "best_val_loss"],
    ascending=[False, True]
).index[0]

best_experiment = results_df.loc[best_index]

print("Meilleure expérience :")
display(best_experiment)

best_model_name = best_experiment["model_name"]
best_model_path = best_experiment["model_path"]

if best_model_name == "MLPSequential":
    best_model = MLPSequential(input_dim=input_dim, num_classes=num_classes)
elif best_model_name == "MLPCustom":
    best_model = MLPCustom(input_dim=input_dim, num_classes=num_classes)
else:
    raise ValueError("Nom de modèle inconnu.")

best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model = best_model.to(device)

test_loss, test_accuracy, test_predictions, test_labels = evaluate(
    best_model,
    test_loader,
    criterion,
    device
)

test_precision = precision_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

print("Résultats sur le test set :")
print(f"Test loss      : {test_loss:.4f}")
print(f"Accuracy       : {test_accuracy:.4f}")
print(f"Precision macro: {test_precision:.4f}")
print(f"Recall macro   : {test_recall:.4f}")
print(f"F1-score macro : {test_f1:.4f}")

print("\nClassification report :")
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(test_labels, test_predictions)

plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)
disp.plot(cmap="Blues", xticks_rotation=45, values_format="d")
plt.title("Matrice de confusion - Meilleur MLP")
plt.tight_layout()

confusion_matrix_path = os.path.join(RESULTS_DIR, "mlp_spotify_confusion_matrix.png")
plt.savefig(confusion_matrix_path, dpi=300)
plt.show()

print("Matrice de confusion sauvegardée :", confusion_matrix_path)

## 9. Inspection PyTorch du meilleur modèle

In [ ]:
print("Paramètres nommés du modèle :")
for name, param in best_model.named_parameters():
    print(name, param.shape, "trainable =", param.requires_grad)

print("\nstate_dict du modèle :")
# state_dict contient les poids et biais appris par le modèle.
for key, value in best_model.state_dict().items():
    print(key, value.shape)

total_trainable_params = sum(
    param.numel()
    for param in best_model.parameters()
    if param.requires_grad
)

print("\nNombre total de paramètres entraînables :", total_trainable_params)

model_device = next(best_model.parameters()).device
print("Device utilisé par le modèle :", model_device)

# Le device cuda/cpu permet d'utiliser le GPU si disponible, sinon le processeur.
# Le meilleur modèle est sauvegardé pour pouvoir le recharger sans refaire tout l'entraînement.